# Simple Inference Notebook

Load a trained checkpoint from `models/checkpoints`, restore the model, and run a single-image prediction.

In [ ]:
import sys
from pathlib import Path
import json
import torch
from PIL import Image
from torchvision import transforms

# Discover repo root (assumes this notebook is executed from the repo or a parent)
def discover_repo_root():
    candidates = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
    for candidate in candidates:
        if (candidate / 'engine').exists():
            return candidate
    return Path.cwd().resolve()

REPO_ROOT = discover_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from engine import config as cfg
from engine.model_factory import build_model

# Ensure we point to the same output folder used during training
checkpoint_dir = Path(getattr(cfg, 'MODEL_OUTPUT_DIR', REPO_ROOT / 'models' / 'checkpoints'))
if not checkpoint_dir.exists():
    raise FileNotFoundError(f'Checkpoint directory not found: {checkpoint_dir}')

# Load class mapping saved by the training notebook
classes_file = REPO_ROOT / 'results' / 'classes.json'
if not classes_file.exists():
    raise FileNotFoundError(f'Classes file not found: {classes_file} -- run the training notebook first')
classes = json.loads(classes_file.read_text())
num_classes = len(classes)
print(f'Found {num_classes} classes')

In [ ]:
# Pick the latest checkpoint in the directory
checkpoint_files = sorted(checkpoint_dir.glob('*.pth'))
if not checkpoint_files:
    raise FileNotFoundError(f'No .pth files found in {checkpoint_dir}')
checkpoint_path = checkpoint_files[-1]
print(f'Using checkpoint: {checkpoint_path.name}')

# Build model matching the checkpoint (assumes model_type string is encoded in filename)
# Fallback: use config.MODEL_TYPE
model_type = getattr(cfg, 'MODEL_TYPE', 'custom_cnn')
pretrained = False
if 'pretrained' in checkpoint_path.name:
    pretrained = True
model = build_model(model_type=model_type, num_classes=num_classes, pretrained=pretrained)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
print('Model loaded.')

In [ ]:
# Define preprocessing (must match validation transforms used during training)
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cfg.IMAGENET_MEAN, std=cfg.IMAGENET_STD)
])

def predict_image(image_path, topk=3):
    img = Image.open(image_path).convert('RGB')
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.nn.functional.softmax(logits, dim=1)[0]
        topk_vals, topk_idx = torch.topk(probs, k=min(topk, len(classes)))
        return [(classes[i], float(topk_vals[j])) for j, i in enumerate(topk_idx)]

# Example: replace with your image path or upload via the notebook UI
example_images = list((REPO_ROOT / 'data' / 'processed').glob('**/*.jpg'))[:5]
if example_images:
    img_path = example_images[0]
    print('Example image:', img_path)
    print(predict_image(img_path, topk=3))
else:
    print('No example images found under data/processed. Upload an image or point to a test file.')